In [ ]:
import os ##### with thesis-seg ENV
import matplotlib.pyplot as plt
from skimage.transform import AffineTransform, SimilarityTransform, warp, resize
from skimage.color import rgb2gray, rgb2hed
from skimage import exposure
from skimage.restoration import denoise_tv_chambolle
from scipy.interpolate import interp1d
from tifffile import imread, imwrite, TiffFile
import colour
import numpy as np


## Loading Images

In [ ]:
def load_HnE_image(folder_path):

    img_data = []
    label_data = []
    
    for _, _, files in os.walk(folder_path):
        for index, file in enumerate(files):
            if file.endswith(".tif"): 
                image_path = os.path.join(folder_path, file)
                img_raw = imread(image_path)
                img = np.array(img_raw)[:, :]
                img_data.append(img)

    return img_data

def display_image(img):
    img = img.astype(float)
    img = img / img.max()
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.colorbar()                 
    plt.show()

In [ ]:
# loading DAPI ROI1
folder_path_dapi = "../../../Thesis_Data/MACSima_HnE_CRC/MACSima_HnE_CRC/CRC027_DAPI"
images_data = load_HnE_image(folder_path_dapi)

In [ ]:
roi_1_dapi_init = images_data[1]
roi_2_dapi_init = images_data[2]
roi_3_dapi_init = images_data[3]
display_image(roi_1_dapi_init)
display_image(roi_2_dapi_init)
display_image(roi_3_dapi_init)

In [ ]:
# loading H&E ROI1
folder_path_hne = "../../../Thesis_Data/MACSima_HnE_CRC/MACSima_HnE_CRC/H&E Scans/CRC027"
images_data_hne = load_HnE_image(folder_path_hne)

In [ ]:
roi_1_hne_init = images_data_hne[0]
roi_2_hne_init = images_data_hne[1]
roi_3_hne_init = images_data_hne[2]
display_image(roi_1_hne_init)
display_image(roi_2_hne_init)
display_image(roi_3_hne_init)

## Preprocessing Images

### H&E Image

##### Initial pre-processing

In [ ]:
# preprocess H&E image (grayscale and contrast enhancement)
gray_image = rgb2gray(roi_1_hne_init)
display_image(gray_image)

image = gray_image.astype(float)
gray_image = image / image.max()
gray_enhanced_hne_roi_1 = exposure.equalize_adapthist(gray_image, clip_limit=0.04)
display_image(gray_enhanced_hne_roi_1)

gray_enhanced_hne_roi_1 = 1 - gray_enhanced_hne_roi_1
display_image(gray_enhanced_hne_roi_1)

##### Valis type pre-processing

In [ ]:
# Applying VALIS style preprocessing to H&E image
eps = np.finfo("float").eps
with colour.utilities.suppress_warnings(colour_usage_warnings=True):
    if 1 < roi_1_hne_init.max() <= 255 and np.issubdtype(roi_1_hne_init.dtype, np.integer):
        cam = colour.convert(roi_1_hne_init/roi_1_hne_init.max() + eps, 'sRGB', 'CAM16UCS')
    else:
        cam = colour.convert(roi_1_hne_init + eps, 'sRGB', 'CAM16UCS')

lum = cam[..., 0] # keep luminosity
cc = np.full_like(lum, 0.2) # colour intensity (chroma)
hc = np.full_like(lum, 0) # hue
new_a, new_b = cc * np.cos(hc), cc * np.sin(hc)
new_cam = np.dstack([lum, new_a+eps, new_b+eps])
with colour.utilities.suppress_warnings(colour_usage_warnings=True):
    rgb2 = colour.convert(new_cam, 'CAM16UCS', 'sRGB')
    rgb2 -= eps

# rgb2 = (np.clip(rgb2, 0, 1)*255).astype(np.uint8)
display_image(rgb2)

gray_image_valis = rgb2gray(rgb2)
gray_image_valis = 1 - gray_image_valis
display_image(gray_image_valis)


In [ ]:
def normalize_image(img, target_percentiles=[0.0, 0.05, 0.5, 0.95], target_values=[0.0, 0.05, 0.5, 0.95]):
    p = np.percentile(img, [v*100 for v in target_percentiles])
    f = interp1d(p, target_values, kind='cubic', fill_value="extrapolate")
    img_norm = f(img)
    return np.clip(img_norm, 0, 1)
    
roi_1_hne_normalized = normalize_image(gray_image_valis)
display_image(roi_1_hne_normalized)

# TV denoising
roi_1_hne_valis = denoise_tv_chambolle(roi_1_hne_normalized, weight=0.01)
display_image(roi_1_hne_valis)

##### VALIS pre-processing

In [ ]:
###### using thesis-seg-valis ENV

from valis.preprocessing import standardize_colorfulness, norm_img_stats, create_tissue_mask, collect_img_stats
from valis import registration

In [ ]:
# preprocess H&E image using VALIS functions
hne_first = standardize_colorfulness(roi_1_hne_init, c=0.2, h=0)
hne_gray_inverted = 1 - rgb2gray(hne_first)
display_image(hne_gray_inverted)

In [ ]:
_, hne_mask = create_tissue_mask(roi_1_hne_init)
_, dapi_img_stats = collect_img_stats(roi_1_dapi_enhanced)
hne_roi_1_VALIS = norm_img_stats(hne_gray_inverted*255, mask=hne_mask, target_stats=dapi_img_stats)
display_image(hne_roi_1_VALIS)

In [ ]:
hne_mask = hne_roi_1_VALIS.copy()
hne_mask[hne_mask >= 100] = 255
hne_mask[hne_mask < 100] = 0
display_image(hne_mask)

##### Colour Deconvolusion

In [ ]:
print("RGB values:", roi_1_hne_init[2061, 1568, :])
print("RGB values:", roi_1_hne_init[2091, 1470, :])
print("RGB values:", roi_1_hne_init[2093, 1472, :])
print("RGB values:", roi_1_hne_init[2341, 1554, :])
print("RGB values:", roi_1_hne_init[2036, 1460, :])


In [ ]:
hed = rgb2hed(roi_1_hne_init)
hematoxylin = hed[:, :, 0]
# display_image(hematoxylin)
hematoxylin_processed = hematoxylin.copy()
hematoxylin_processed[hematoxylin_processed < 0.03] = 0
hematoxylin_processed[hematoxylin_processed >= 0.03] = 1
display_image(hematoxylin_processed)
"""
hne_nuclei_extracted = roi_1_hne_init.copy()
hne_nuclei_extracted[:,:,0][hne_nuclei_extracted[:,:,0] < 200] = 255
hne_nuclei_extracted[:,:,1][hne_nuclei_extracted[:,:,1] < 170] = 255
hne_nuclei_extracted[:,:,2][hne_nuclei_extracted[:,:,2] < 150] = 255
display_image(hne_nuclei_extracted)
"""

hne_nuclei_extracted = roi_1_hne_init.copy()
hne_nuclei_extracted[(hne_nuclei_extracted[:,:,0] < 200) & (hne_nuclei_extracted[:,:,1] > 20) & (hne_nuclei_extracted[:,:,2] < 150)] = [255, 255, 255]
display_image(hne_nuclei_extracted)
gray_nuclei_extracted = 1- rgb2gray(hne_nuclei_extracted)
display_image(gray_nuclei_extracted)

In [ ]:
hne_mask = hematoxylin_processed.copy()
hne_mask[hne_mask >= 0.02] = 1
display_image(hne_mask)

### DAPI Image

##### Initial pre-processing

In [ ]:
# preprocess DAPI image 
image = roi_1_dapi_init.astype(float)
scale = 0.5023/0.209877
roi_1_dapi_init_scaled = resize(image, (int(image.shape[0]/scale), int(image.shape[1]/scale)), anti_aliasing=True)
roi_1_dapi = roi_1_dapi_init_scaled / roi_1_dapi_init_scaled.max()
display_image(roi_1_dapi)
roi_1_dapi_enhanced = exposure.equalize_adapthist(roi_1_dapi, clip_limit=0.03)
display_image(roi_1_dapi_enhanced)

##### Valis type preprocessing

In [ ]:
gray_normalized = normalize_image(roi_1_dapi)
display_image(gray_normalized)

# TV denoising
roi_1_dapi_valis = denoise_tv_chambolle(gray_normalized, weight=0.01)
display_image(roi_1_dapi_valis)

##### VALIS pre-processing

In [ ]:
# preprocess DAPI image using VALIS functions
dapi_roi_1_VALIS = norm_img_stats(roi_1_dapi_enhanced*255, target_stats=dapi_img_stats)
display_image(dapi_roi_1_VALIS)

In [ ]:
dapi_mask = dapi_roi_1_VALIS.copy()
dapi_mask[dapi_mask >= 100] = 255
dapi_mask[dapi_mask < 100] = 0
display_image(dapi_mask)

## Pre-processing Cropped Images

In [ ]:
hne_cropped = imread("../../../Thesis_Data/for_valis/CRC027/cropped/hne_cropped.tif")
display_image(hne_cropped)
dapi_cropped = imread("../../../Thesis_Data/for_valis/CRC027/cropped/dapi_cropped.tif")
display_image(dapi_cropped)

## Registration using ORB, RANSAC

In [ ]:
# using ORB and RANSAC for registration

import matplotlib.pyplot as plt
from skimage.feature import ORB, match_descriptors, plot_matched_features
from skimage.transform import AffineTransform, warp, resize, SimilarityTransform
from skimage.measure import ransac

# Resize images to be the same size
# image1 = resize(image1, (500, 500), anti_aliasing=True)
# image2 = resize(image2, (500, 500), anti_aliasing=True)

# Resize the images 
roi_1_dapi_scaled = resize(dapi_mask, (roi_1_dapi_init_scaled.shape[0] // 2, roi_1_dapi_init_scaled.shape[1] // 2), anti_aliasing=True)
gray_enhanced_hne_roi_1_scaled = resize(hematoxylin_processed, (roi_1_hne_init.shape[0] // 2, roi_1_hne_init.shape[1] // 2), anti_aliasing=True)

# Initialize ORB detector
orb = ORB(n_keypoints=1000, fast_threshold=0.05)

# Detect features and descriptors
orb.detect_and_extract(gray_enhanced_hne_roi_1_scaled)
keypoints1 = orb.keypoints
descriptors1 = orb.descriptors

orb.detect_and_extract(roi_1_dapi_scaled)
keypoints2 = orb.keypoints
descriptors2 = orb.descriptors

# Match descriptors
matches = match_descriptors(descriptors1, descriptors2, cross_check=True)

# Extract matched keypoints
src = keypoints1[matches[:, 0]]
dst = keypoints2[matches[:, 1]]

# Plot keypoints and matches
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
plt.gray()
plot_matched_features(gray_enhanced_hne_roi_1_scaled, roi_1_dapi_scaled, keypoints0=keypoints1, keypoints1=keypoints2, matches=matches, ax=ax)
ax.axis('off')
ax.set_title("Keypoint Matches")
plt.show()

# Verify number of matches
print(f"Number of matches: {len(matches)}")

# Check if we have enough matches to compute a reliable transformation
if len(matches) < 4:
    raise ValueError("Not enough matches to compute a reliable transformation")

# Compute affine transformation using RANSAC for robustness
np.random.seed(42) 
model_robust, inliers = ransac((dst, src),
                               AffineTransform, min_samples=4,
                               residual_threshold=2, max_trials=1000)

# Warp image
registered_image = warp(gray_enhanced_hne_roi_1_scaled, model_robust.inverse, output_shape=roi_1_dapi_scaled.shape)

# Display results
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
plt.title('HnE ROI 1')
plt.imshow(gray_enhanced_hne_roi_1_scaled, cmap='gray')

plt.subplot(1, 3, 2)
plt.title('DAPI ROI 1')
plt.imshow(roi_1_dapi_scaled, cmap='gray')

plt.subplot(1, 3, 3)
plt.title('Registered Image (aligned version of HnE ROI 1)')
plt.imshow(registered_image, cmap='gray')

plt.show()

## Registration using SIFT, RANSAC

In [ ]:
# using SIFT and RANSAC for registration

import matplotlib.pyplot as plt

from skimage.transform import AffineTransform, warp, resize
from skimage.color import rgb2gray
from skimage.feature import match_descriptors, plot_matched_features, SIFT

# Resize the images to reduce memory usage
roi_1_dapi_scaled = resize(roi_1_dapi_valis, (roi_1_dapi_valis.shape[0] // 8, roi_1_dapi_valis.shape[1] // 8), anti_aliasing=True).astype(np.float32)
gray_enhanced_hne_roi_1_scaled = resize(roi_1_hne_valis, (roi_1_hne_valis.shape[0] // 4, roi_1_hne_valis.shape[1] // 4), anti_aliasing=True).astype(np.float32)

# Resize images to be the same size
# img1 = resize(image1, (500, 500), anti_aliasing=True)
# img2 = resize(image2, (500, 500), anti_aliasing=True)

descriptor_extractor = SIFT(n_octaves=3, n_scales=5)

descriptor_extractor.detect_and_extract(gray_enhanced_hne_roi_1_scaled)
keypoints1 = descriptor_extractor.keypoints
descriptors1 = descriptor_extractor.descriptors

descriptor_extractor.detect_and_extract(roi_1_dapi_scaled)
keypoints2 = descriptor_extractor.keypoints
descriptors2 = descriptor_extractor.descriptors

matches12 = match_descriptors(
    descriptors1, descriptors2, max_ratio=0.4, cross_check=True
)

# Extract matched keypoints
src = keypoints1[matches12[:, 0]]
dst = keypoints2[matches12[:, 1]]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11, 8))

plt.gray()

plot_matched_features(
    gray_enhanced_hne_roi_1_scaled,
    roi_1_dapi_scaled,
    keypoints0=keypoints1,
    keypoints1=keypoints2,
    matches=matches12,
    ax=ax[0],
)
ax[0].axis('off')
ax[0].set_title("H&E vs. DAPI\n" "(all keypoints and matches)")


plot_matched_features(
    gray_enhanced_hne_roi_1_scaled,
    roi_1_dapi_scaled,
    keypoints0=keypoints1,
    keypoints1=keypoints2,
    matches=matches12[::15],
    ax=ax[1],
    only_matches=True,
)
ax[1].axis('off')
ax[1].set_title(
    "H&E vs. DAPI\n" "(subset of matches for visibility)"
)


plt.tight_layout()
plt.show()


# Compute affine transformation using RANSAC for robustness
model_robust, inliers = ransac((dst, src),
                               AffineTransform, min_samples=4,
                               residual_threshold=2, max_trials=1000)

# Warp image
registered_image = warp(gray_enhanced_hne_roi_1_scaled, model_robust.inverse, output_shape=roi_1_dapi_scaled.shape)

# Display results
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
plt.title('H&E ROI 1')
plt.imshow(gray_enhanced_hne_roi_1_scaled, cmap='gray')

plt.subplot(1, 3, 2)
plt.title('DAPI ROI 1')
plt.imshow(roi_1_dapi_scaled, cmap='gray')

plt.subplot(1, 3, 3)
plt.title('Registered Image (aligned version of H&E ROI 1)')
plt.imshow(registered_image, cmap='gray')

plt.show()

## Registration using VALIS

In [ ]:
# VALIS for WSI registration
from valis import registration, data
slide_src_dir = "../../../Thesis_Data/for_valis/CRC027/hne"
results_dst_dir = "../../../Thesis_Data/for_valis/CRC027/results_hne"
registered_slide_dst_dir = "../../../Thesis_Data/for_valis/CRC027/registered_hne"

registrar = registration.Valis(slide_src_dir, results_dst_dir)
rigid_registrar, non_rigid_registrar, error_df = registrar.register()
#registrar.warp_and_save_slides(registered_slide_dst_dir)

registration.kill_jvm()

In [ ]:
# imwrite("../../../Thesis_Data/for_valis/CRC027/input_images/hne.tiff", hne_roi_1_VALIS)
# imwrite("../../../Thesis_Data/for_valis/CRC027/input_images/dapi.tiff", dapi_VALIS_rescaled)
imwrite("../../../Thesis_Data/for_valis/CRC027/input_images/hne.tiff", hematoxylin_processed)
imwrite("../../../Thesis_Data/for_valis/CRC027/input_images/dapi.tiff", roi_1_dapi_init)

In [ ]:
# VALIS for ROI registration
from valis import registration

slide_src_dir = "../../../Thesis_Data/for_valis/CRC027/input_images"
results_dst_dir = "../../../Thesis_Data/for_valis/CRC027/results"
registered_slide_dst_dir = "../../../Thesis_Data/for_valis/CRC027/registered"


reg = registration.serial_rigid.register_images(
    slide_src_dir,
    results_dst_dir,  
)

# non_rigid_registrar = serial_non_rigid.register_images(rigid_registrar)

## Using Optical Flow methods for registration

#### Using tvl

In [ ]:
# DAPI as moving image

from skimage.registration import optical_flow_ilk, optical_flow_tvl1 
import cv2 as cv 
from skimage.transform import warp, resize 
import scipy as sp 
import numpy as np 

def apply_optical_flow(temp, moving):
    rescaled_temp = resize(temp, (3025, 4512), anti_aliasing=True)
    rescaled_moving = resize(moving, (3025, 4512), anti_aliasing=True)
    template = rescaled_temp
    moved = rescaled_moving

    # registration 
    """
    v, u = optical_flow_ilk(template, moved, radius=10, prefilter=False, num_warp=15, gaussian=True) 
    nr, nc = template.shape 
    row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij') 
    registered= warp(moved, np.array([row_coords + v, col_coords + u]), mode='constant', preserve_range=True) 
    """

    v, u = optical_flow_tvl1(template, moved, attachment=100, tightness=1.20, 
                              num_warp=15, num_iter=15, tol=0.1, prefilter=True)
    nr, nc = template.shape
    row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')
    registered= warp(moved, np.array([row_coords + v, col_coords + u]), 
                 mode='constant', preserve_range=True, order=0)

    pearson_corr_R = sp.stats.pearsonr(template.flatten(), moved.flatten())[0] 
    pearson_corr_R_reg = sp.stats.pearsonr(template.flatten(), registered.flatten())[0] 
    print(f"Pearson correlation coefficient image vs. moved: {pearson_corr_R}") 
    print(f"Pearson correlation coefficient image vs. registration: {pearson_corr_R_reg}")

    return template, moved, registered, v, u

In [ ]:
template, moved, registered, v, u = apply_optical_flow(roi_1_hne_valis, roi_1_dapi_valis)

In [ ]:
display_image(template)
display_image(moved)
display_image(registered)

In [ ]:
# Registered DAPI image
dapi_rescaled = resize(roi_1_dapi_init, (3025, 4512), anti_aliasing=True)
nr, nc = template.shape
row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')
registered= warp(dapi_rescaled, np.array([row_coords + v, col_coords + u]), 
                 mode='constant', preserve_range=True, order=0)

display_image(registered)
display_image(roi_1_dapi_init)

In [ ]:
# H&E as moving image
template, moved, registered, v, u = apply_optical_flow(roi_1_dapi_valis, roi_1_hne_valis)
display_image(template)
display_image(moved)
display_image(registered)

In [ ]:
# registered H&E image
registered = np.zeros_like(roi_1_hne_init)
for c in range(3):
    registered[..., c] = warp(
        roi_1_hne_init[..., c],
        np.array([row_coords + v, col_coords + u]),
        mode='constant',
        preserve_range=True,
        order=0
    )

display_image(registered)
display_image(roi_1_hne_init)

#### Using ilk

In [ ]:
from skimage.registration import optical_flow_ilk, optical_flow_tvl1 
import cv2 as cv 
from skimage.transform import warp, resize 
import scipy as sp 
import numpy as np 

dapi_rescaled = resize(dapi_roi_1_VALIS, (3025, 4512), anti_aliasing=True)
template = hne_roi_1_VALIS 
moved = dapi_rescaled

# registration 

v, u = optical_flow_ilk(template, moved, radius=10, prefilter=False, num_warp=15, gaussian=True) 
nr, nc = template.shape 
row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij') 
registered= warp(moved, np.array([row_coords + v, col_coords + u]), mode='constant', preserve_range=True) 


pearson_corr_R = sp.stats.pearsonr(template.flatten(), moved.flatten())[0] 
pearson_corr_R_reg = sp.stats.pearsonr(template.flatten(), registered.flatten())[0] 
print(f"Pearson correlation coefficient image vs. moved: {pearson_corr_R}") 
print(f"Pearson correlation coefficient image vs. registration: {pearson_corr_R_reg}")

display_image(template)
display_image(moved)
display_image(registered)

## Using Phase Cross-correlation for registration

In [ ]:
# A default registration pipeline with phase_cross_corr(): 

from skimage.registration import phase_cross_correlation 
from skimage.transform import SimilarityTransform, warp, rotate, warp_polar 
import scipy as sp 

template = hne_roi_1_VALIS 
moved = dapi_rescaled 

# registration: 
"""
shift, _, _ = phase_cross_correlation(template, moved, upsample_factor=30, normalization="phase") 
shifts=[shift[1], shift[0]] 
print(f'Detected translational shift: {shifts}') 
tform = SimilarityTransform(translation=(-shift[1], -shift[0])) 
registered = warp(moved, tform, preserve_range=True) 
"""

radius_polar=800 
template_polar = warp_polar(template, radius=radius_polar, preserve_range=True) 
moved_polar = warp_polar(moved, radius=radius_polar, preserve_range=True) 
shift_rot, _, _ = phase_cross_correlation(template_polar, moved_polar, upsample_factor=30, normalization="phase") 
shift_rot = shift_rot[0] 
print(f'Detected counterclockwise rotation: {shift_rot}°') 
registered = rotate(moved, -shift_rot, preserve_range=True)

# metrics: 

pearson_corr_R = sp.stats.pearsonr(template.flatten(), moved.flatten())[0] 
pearson_corr_R_reg = sp.stats.pearsonr(template.flatten(), registered.flatten())[0] 
print(f"Pearson correlation coefficient image vs. moved: {pearson_corr_R}") 
print(f"Pearson correlation coefficient image vs. registration: {pearson_corr_R_reg}")

display_image(template)
display_image(moved)        
display_image(registered)